# Table extraction sample (Pietro)

Reconstructed from `sample_code_pietro.py` (Colab .py export).
Original Colab header:

```
Untitled10.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1s7Srg0_lR_v4pTduxouHqkwOkuFZzout
```

In [5]:
import os

In [ ]:
# Install necessary libraries
!pip install -qqq transformers datasets "layoutparser[ocr]" pdf2image Pillow
# Install detectron2 separately as it requires a specific command
!pip install -qqq "git+https://github.com/facebookresearch/detectron2.git"

# Install poppler-utils for pdf2image to work (if not already installed in Colab env)
!apt-get update -qqq # Update apt-get first
!apt-get install -qqq poppler-utils

ERROR: Failed to build 'git+https://github.com/facebookresearch/detectron2.git' when getting requirements to build wheel
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import importlib, subprocess, sys

# Install any missing packages before importing
for module, pip_name in [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("PIL", "Pillow"),
    ("layoutparser", "layoutparser[ocr]"),
    ("pdf2image", "pdf2image"),
]:
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qqq", pip_name])

import torch
# Use AutoImageProcessor and AutoModelForObjectDetection for structure recognition model
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image
import layoutparser as lp
from pdf2image import convert_from_path

print("Libraries installed and imported successfully!")

In [ ]:
# Load the image processor and model for structure recognition
model_name = "microsoft/table-transformer-structure-recognition"
image_processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForObjectDetection.from_pretrained(model_name)

print(f"Model '{model_name}' and image processor loaded successfully!")

In [ ]:
# Ensure model is on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model moved to {device}")

In [ ]:
# --- Next Steps ---
# 1. Upload your PDF file to your Colab environment (e.g., drag and drop to the files section).
# 2. Provide the path to your PDF file below.
# 3. We will then write code to convert the PDF pages into images and process them.
# For example, if your PDF is named 'my_document.pdf':
pdf_path = "/content/AZ_PHOENIXCITY-COPERS_AV_2019_94.pdf"

# Convert PDF pages into images
print(f"Converting pages from {pdf_path}...")
pages = convert_from_path(pdf_path, 300) # 300 DPI for good quality
print(f"Converted {len(pages)} pages from PDF.")

### Table Extraction Process

1.  **Iterate through PDF pages**: Each page of the PDF is now an image.
2.  **Model Prediction**: For each image, the `TableTransformerForObjectDetection` model will detect table regions.
3.  **Layout Parsing**: `layoutparser` will be used to process the detected table regions and perform OCR.
4.  **Extract Table Headers**: We will then focus on extracting text from detected table headers.

In [ ]:
import json

# Initialize layoutparser for OCR
ocr_agent = lp.TesseractAgent(languages='eng') # English language for OCR

all_extracted_tables = []

for i, page_image in enumerate(pages):
    print(f"Processing page {i+1}/{len(pages)}...")

    # Prepare image for the model
    inputs = image_processor(images=page_image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to GPU if available

    # Make predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert outputs to COCO API's format
    target_sizes = torch.tensor([page_image.size[::-1]])
    results = image_processor.post_process_object_detection(outputs, threshold=0.9, target_sizes=target_sizes)[0]

    # Filter for tables (class_id 0)
    table_detections = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        if label == 0: # Table class
            x, y, x2, y2 = box.tolist()
            table_detections.append({'box': [x, y, x2, y2], 'score': score.item(), 'label': label.item()})

    # Process each detected table
    for table_idx, table_info in enumerate(table_detections):
        table_box = [int(coord) for coord in table_info['box']]
        cropped_table_image = page_image.crop(table_box)

        # Re-run model inference on the cropped table image to get internal structure (cells, headers)
        table_inputs = image_processor(images=cropped_table_image, return_tensors="pt")
        table_inputs = {k: v.to(device) for k, v in table_inputs.items()}

        with torch.no_grad():
            table_outputs = model(**table_inputs)

        table_target_sizes = torch.tensor([cropped_table_image.size[::-1]])
        # Using a lower threshold for internal elements to catch more cells
        internal_results = image_processor.post_process_object_detection(table_outputs, threshold=0.7, target_sizes=table_target_sizes)[0]

        detected_cells = []
        # Labels for cells: 5 (table spanning cell), 6 (table cell)
        # Labels for headers: 3 (table column header)
        for score, label, box in zip(internal_results["scores"], internal_results["labels"], internal_results["boxes"]):
            if label in [3, 5, 6]: # Consider these as content cells, including headers
                x, y, x2, y2 = box.tolist()
                cell_block = lp.TextBlock(block=lp.Rectangle(x, y, x2, y2), text='', id=f'cell_{len(detected_cells)}', type=model.config.id2label[label.item()])

                # Crop the cell image from the cropped table image
                cell_image = cropped_table_image.crop((int(x), int(y), int(x2), int(y2)))
                cell_text = ocr_agent.detect(cell_image)
                cell_block.text = cell_text.strip() # Remove leading/trailing whitespace

                detected_cells.append({
                    'box': [int(x), int(y), int(x2), int(y2)],
                    'text': cell_block.text,
                    'type': cell_block.type,
                    'score': score.item()
                })

        # Sort cells first by y-coordinate (row), then by x-coordinate (column)
        detected_cells.sort(key=lambda c: (c['box'][1], c['box'][0]))

        if detected_cells:
            all_extracted_tables.append({
                'page': i + 1,
                'table_index': table_idx + 1,
                'bounding_box': table_box,
                'cells': detected_cells
            })
        else:
            # If no cells are detected within the table, OCR the entire table region
            full_table_ocr_text = ocr_agent.detect(cropped_table_image)
            all_extracted_tables.append({
                'page': i + 1,
                'table_index': table_idx + 1,
                'bounding_box': table_box,
                'full_table_ocr_text': full_table_ocr_text.strip() # Store the full OCR text if no cells found
            })

print("\n--- Table Extraction Complete ---")
print(f"Total tables found: {len(all_extracted_tables)}")

In [ ]:
# Output the extracted tables in JSON format
json_output = json.dumps(all_extracted_tables, indent=2)
print("\n--- JSON Output ---")
print(json_output)

### Qwen-VL Table Extraction

In [ ]:
import base64
from io import BytesIO
from PIL import Image

def process_vision_info(messages):
    image_inputs = []
    video_inputs = []
    for message in messages:
        if message["role"] == "user":
            for item in message["content"]:
                if item["type"] == "image":
                    # Assuming item["image"] is already a PIL Image object
                    image_inputs.append(item["image"])
                # Add logic for video if needed
    return image_inputs, video_inputs

print("process_vision_info function defined.")

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

# Load model natively onto your GPU (will take ~14-15GB VRAM)
print("Loading Qwen2.5-VL-7B-Instruct model...")
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Qwen2.5-VL-7B-Instruct model and processor loaded successfully!")

In [ ]:
qwen_extracted_tables_markdown = []

for i, page_image in enumerate(pages):
    print(f"Qwen-VL processing page {i+1}/{len(pages)}...")

    # Request a strict layout/table parse
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image}, # Pass PIL Image directly
                {"type": "text", "text": "Extract all tables from this document image and format them perfectly into a clean markdown format. Do not skip any rows or columns."}
            ]
        }
    ]

    # Process and run inference
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    # Fix: Pass None for videos if video_inputs is empty, to prevent IndexError
    inputs = qwen_processor(text=[text], images=image_inputs, videos=video_inputs if video_inputs else None, padding=True, return_tensors="pt").to(qwen_model.device)

    generated_ids = qwen_model.generate(**inputs, max_new_tokens=4096)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = qwen_processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    qwen_extracted_tables_markdown.append({
        'page': i + 1,
        'markdown_output': output_text[0].strip()
    })
    # print(f"\n--- Qwen-VL Output for Page {i+1} ---\n{output_text[0].strip()}\n")

print("\n--- Qwen-VL Table Extraction Complete ---")
print(f"Total pages processed by Qwen-VL: {len(qwen_extracted_tables_markdown)}")

In [ ]:
# Optionally, print all collected markdown outputs in a single block or save to file
print("\n--- All Qwen-VL Markdown Outputs (Collected) ---")
for result in qwen_extracted_tables_markdown:
    print(f"\nPage {result['page']}:\n{result['markdown_output']}")

# If you want it all as a single JSON structure
json_qwen_output = json.dumps(qwen_extracted_tables_markdown, indent=2)
print("\n--- Qwen-VL JSON Output ---")
print(json_qwen_output)

In [ ]:
# Display the extracted table headers
for table_data in all_extracted_tables:
    print(f"\nPage: {table_data['page']}, Table Index: {table_data['table_index']}")
    if 'headers' in table_data:
        print(f"Headers: {table_data['headers']}")
    elif 'full_table_ocr_text' in table_data:
        print(f"Full Table OCR Text (no specific headers found):\n{table_data['full_table_ocr_text']}")